# Electrode Map Generator

This notebook maps electrode contacts to PCB or headstage channels.

The workflow:

1. Select electrode layout DXF files.
2. Select component mapping JSON files.
3. Select assembly connection CSV files.
4. Generate a complete channel map.
5. Export the result as CSV.

The generated channel map has shape (4 x N), with rows containing:

1. Electrode X-coordinate in micrometres
2. Electrode Y-coordinate in micrometres
3. Destination channel
4. Source electrode instance

In [ ]:
from collections import defaultdict
import tempfile
from pathlib import Path

import ezdxf
from IPython.display import display
import ipywidgets as widgets

import utils
from assembly_routing import AssemblyRouter, ComponentDefinition


In [ ]:
# Optional: inspect the layers in an example DXF file.
example_dxf = Path(
    "../electrode_map_example/example1/50um_pitch_compatible_with_FPC.dxf"
)
doc = ezdxf.readfile(example_dxf)
layers = sorted({entity.dxf.layer for entity in doc.modelspace()})
print("Available DXF layers:")
print(layers)


In [ ]:
# Name of the DXF layer containing electrode contours.
# Set to None when the layer is unknown.
layer_name = "electrodes"

# Used only when layer_name is None. The parser removes the final contour,
# which is assumed to be a hook loop, then selects the requested contours.
num_contours = 64

In [ ]:
dxf_chooser = widgets.FileUpload(
    accept=".dxf",
    multiple=True,
    description="Select electrode DXF files",
)

electrode_chooser = widgets.FileUpload(
    accept=".json",
    multiple=True,
    description="Select electrode mapping JSON",
)

flex_pcb_chooser = widgets.FileUpload(
    accept=".json",
    multiple=True,
    description="Select flex PCB mapping JSON",
)

rigid_pcb_chooser = widgets.FileUpload(
    accept=".json",
    multiple=True,
    description="Select rigid PCB mapping JSON",
)

headstage_chooser = widgets.FileUpload(
    accept=".json",
    multiple=True,
    description="Select headstage mapping JSON",
)

assembly_chooser = widgets.FileUpload(
    accept=".csv",
    multiple=True,
    description="Select assembly CSV file",
)

display(dxf_chooser)
display(electrode_chooser)
display(flex_pcb_chooser)
display(rigid_pcb_chooser)
display(headstage_chooser)
display(assembly_chooser)


In [ ]:
run_button = widgets.Button(
    description="Generate Channel Map",
    button_style="success",
)
output = widgets.Output()

state = {"channel_map": None, "dxf_file": None}


def extract_uploaded_files(upload_widget, temp_dir):
    """Save uploaded widget files to a temporary directory."""
    saved_paths = []

    # Support both ipywidgets v7 (dict) and v8 (tuple) file formats.
    files = upload_widget.value
    if isinstance(files, tuple):
        for file_item in files:
            name = (
                file_item["name"]
                if isinstance(file_item, dict)
                else file_item.name
            )
            content = (
                file_item["content"]
                if isinstance(file_item, dict)
                else file_item.content
            )
            filepath = Path(temp_dir) / name
            filepath.write_bytes(bytes(content))
            saved_paths.append(filepath)
    elif isinstance(files, dict):
        for name, data in files.items():
            filepath = Path(temp_dir) / name
            filepath.write_bytes(bytes(data["content"]))
            saved_paths.append(filepath)

    return saved_paths


def on_run_clicked(button):
    with output:
        output.clear_output(wait=True)

        if not dxf_chooser.value:
            print("Please select at least one DXF file.")
            return

        with tempfile.TemporaryDirectory() as temp_dir:
            dxf_paths = extract_uploaded_files(dxf_chooser, temp_dir)
            assembly_paths = extract_uploaded_files(assembly_chooser, temp_dir)
            electrode_paths = extract_uploaded_files(electrode_chooser, temp_dir)
            flex_pcb_paths = extract_uploaded_files(flex_pcb_chooser, temp_dir)
            rigid_pcb_paths = extract_uploaded_files(rigid_pcb_chooser, temp_dir)
            headstage_paths = extract_uploaded_files(headstage_chooser, temp_dir)

            try:
                print("1. Building Assembly Router & tracing paths...")
                router = AssemblyRouter()

                component_files = [
                    ("electrode", electrode_paths),
                    ("flex_pcb", flex_pcb_paths),
                    ("rigid_pcb", rigid_pcb_paths),
                    ("headstage", headstage_paths),
                ]

                # The last uploaded non-electrode category is the terminal.
                active_non_electrode_categories = [
                    category
                    for category, paths in component_files
                    if category != "electrode" and len(paths) > 0
                ]

                if not active_non_electrode_categories:
                    print(
                        "Please select at least one target component "
                        "(flex_pcb, rigid_pcb, or headstage)."
                    )
                    return

                terminal_category = active_non_electrode_categories[-1]
                print(
                    "   Terminal component category selected: "
                    f"{terminal_category}"
                )

                instance_counts = defaultdict(int)
                component_categories = {}

                for category, paths in component_files:
                    for path in paths:
                        definition = ComponentDefinition.from_json(path)

                        # Avoid registering identical models more than once.
                        if definition.model_name not in router.models:
                            router.register_model(definition)

                        instance_id = f"{category}_{instance_counts[category]}"
                        router.add_instance(instance_id, definition.model_name)

                        component_categories[instance_id] = category
                        instance_counts[category] += 1

                for path in assembly_paths:
                    print(f"   Loading Assembly CSV: {path.name}")
                    router.load_assembly_csv(path)

                starts = []
                electrode_instances = []
                electrode_channel_indices = {}

                for instance_id, category in component_categories.items():
                    if category != "electrode":
                        continue

                    electrode_instances.append(instance_id)

                    model_name = router.instances[instance_id]
                    definition = router.models[model_name]

                    sorted_channels = sorted(
                        definition.mapping,
                        key=lambda channel: (
                            int(str(channel))
                            if str(channel).isdigit()
                            else str(channel)
                        ),
                    )

                    electrode_channel_indices[instance_id] = {
                        str(channel): index
                        for index, channel in enumerate(sorted_channels)
                    }

                    starts.extend(
                        f"{instance_id}.in_{channel}"
                        for channel in sorted_channels
                    )

                report = router.report(
                    starts,
                    terminal_categories=[terminal_category],
                    instance_categories=component_categories,
                )

                complete_report = [
                    item for item in report if item["status"] == "complete"
                ]

                if not complete_report:
                    print("No complete signal paths discovered by Assembly Router.")
                    return

                electrode_instances.sort(key=lambda x: int(x.split("_")[1]))

                if len(dxf_paths) < len(electrode_instances):
                    print(
                        f"Warning: Uploaded {len(dxf_paths)} DXF file(s) for "
                        f"{len(electrode_instances)} electrode instance(s). "
                        "Reusing the last DXF file for remaining electrodes."
                    )

                print("2. Loading electrode DXF layouts...")
                electrode_xy_maps = {}
                electrode_offsets = {}
                running_offset = 0

                for index, instance_id in enumerate(electrode_instances):
                    dxf_path = (
                        dxf_paths[index]
                        if index < len(dxf_paths)
                        else dxf_paths[-1]
                    )
                    print(f"   Mapping {instance_id} -> {dxf_path.name}")

                    xy = utils.load_and_plot_electrodes(
                        dxf_path,
                        layer_name=layer_name,
                        num_contours=num_contours,
                    )

                    electrode_xy_maps[instance_id] = xy
                    electrode_offsets[instance_id] = running_offset
                    running_offset += len(xy)

                total_electrodes = running_offset

                print("3. Generating channel map matrix...")
                channel_map = utils.build_channel_map(
                    complete_report,
                    electrode_channel_indices,
                    electrode_offsets,
                    electrode_xy_maps,
                    total_electrodes,
                )

                state["channel_map"] = channel_map
                state["dxf_file"] = str(dxf_paths[0])

                print("\nSuccess!")
                print(f"Total Electrodes Registered: {len(electrode_instances)}")
                print(f"Channel Map Shape: {channel_map.shape}")
                print(f"Sample column 0: {channel_map[:, 0]}")

            except Exception as error:
                print(f"Error generating channel map: {error}")


## Generate channel map

In [ ]:
run_button.on_click(on_run_clicked)
display(run_button, output)

In [ ]:
export_button = widgets.Button(
    description="Export to CSV",
    button_style="info",
)
export_result = widgets.Output()


def export_channel_map(button):
    """Export the generated channel map to a CSV file."""
    with export_result:
        export_result.clear_output(wait=True)

        if state["channel_map"] is None:
            print("Generate a channel map first")
            return

        try:
            export_file = utils.export_channel_map_to_csv(
                state["channel_map"],
                state["dxf_file"],
            )
            print(f"Exported to: {export_file}")
        except Exception as error:
            print(f"Export failed: {error}")


export_button.on_click(export_channel_map)
display(export_button, export_result)
